PPO / REINFORCE — One Step of Optimization with a Tiny LLM
===========================================================
Purpose: make every tensor in the policy gradient update visible.

Architecture
------------
- Vocabulary: 8 tokens  (think of them as words: [PAD, a, b, c, d, e, f, EOS])
- Embedding dim: 16
- Transformer: 1 layer, 2 heads, ffn_dim=32
- Max sequence length: 6 tokens

Scenario
--------
- Prompt  : [1, 2]          (tokens "a b")
- Generate: up to 4 tokens  (the LLM's "response")
- Reward  : a simple rule-based function (verifiable reward, no RM needed)
            +1 for each token == 3 ("c") in the response
            This keeps the reward honest and inspectable.

Steps shown
-----------
1. Forward pass & sampling
2. Log-probabilities of the sampled tokens
3. Reward computation
4. REINFORCE gradient (no baseline)
5. Baseline subtraction  →  why variance drops
6. One gradient step on θ
7. Verify: log-prob of "good" tokens went up
"""

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

In [5]:
# ─────────────────────────────────────────────
# 0. Vocabulary & helpers
# ─────────────────────────────────────────────
VOCAB   = {0: "PAD", 1: "a", 2: "b", 3: "c", 4: "d", 5: "e", 6: "f", 7: "EOS"}
V       = len(VOCAB)       # 8
D       = 16               # embedding dim
MAX_LEN = 10               # positional encoding size
SEP     = "─" * 60

def tokens_to_str(ids):
    return " ".join(VOCAB[i] for i in ids)

In [6]:
# ─────────────────────────────────────────────
# 1. Tiny Transformer LM
# ─────────────────────────────────────────────
class TinyLM(nn.Module):
    def __init__(self, vocab_size=V, d_model=D, nhead=2, ffn_dim=32, max_len=MAX_LEN):
        super().__init__()
        self.embed    = nn.Embedding(vocab_size, d_model)
        self.pos_enc  = nn.Embedding(max_len, d_model)   # learned positions
        layer         = nn.TransformerEncoderLayer(
                            d_model=d_model, nhead=nhead,
                            dim_feedforward=ffn_dim,
                            batch_first=True, dropout=0.0)
        self.tf       = nn.TransformerEncoder(layer, num_layers=1)
        self.head     = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids):
        """
        token_ids : (1, T)  — single sequence, no batch dim complexity
        returns   : (1, T, V)  logits over vocab at every position
        """
        T    = token_ids.shape[1]
        pos  = torch.arange(T).unsqueeze(0)          # (1, T)
        x    = self.embed(token_ids) + self.pos_enc(pos)

        # causal mask: token t can only attend to 0..t
        mask = nn.Transformer.generate_square_subsequent_mask(T)
        x    = self.tf(x, mask=mask, is_causal=True)
        return self.head(x)                           # (1, T, V)

model = TinyLM()
print(SEP)
print("MODEL")
print(SEP)
total_params = sum(p.numel() for p in model.parameters())
print(f"  Parameters : {total_params}")
print(f"  Vocab size : {V}  ({list(VOCAB.values())})")
print(f"  d_model    : {D}")
print()

────────────────────────────────────────────────────────────
MODEL
────────────────────────────────────────────────────────────
  Parameters : 2648
  Vocab size : 8  (['PAD', 'a', 'b', 'c', 'd', 'e', 'f', 'EOS'])
  d_model    : 16



In [7]:
# ─────────────────────────────────────────────
# 2. Prompt
# ─────────────────────────────────────────────
prompt     = [1, 2]          # "a b"
MAX_GEN    = 4               # generate up to 4 response tokens
EOS_ID     = 7

print(SEP)
print("STEP 1 — PROMPT & SAMPLING")
print(SEP)
print(f"  Prompt tokens : {prompt}  →  '{tokens_to_str(prompt)}'")
print()

────────────────────────────────────────────────────────────
STEP 1 — PROMPT & SAMPLING
────────────────────────────────────────────────────────────
  Prompt tokens : [1, 2]  →  'a b'



In [8]:
# ─────────────────────────────────────────────
# 3. Autoregressive sampling  (store log-probs)
# ─────────────────────────────────────────────
# We'll keep a list of (token_sampled, log_prob_of_that_token)
# so we can assemble the policy gradient later.

generated_tokens  = []    # response tokens only
log_probs_sampled = []    # log π_θ(a_t | s_t)  for each response token
all_logits_list   = []    # for inspection

current_ids = torch.tensor([prompt])   # (1, len(prompt))

model.eval()   # no dropout during generation (but we keep grad for log-probs)

for step in range(MAX_GEN):
    logits = model(current_ids)          # (1, T_current, V)
    # Predict the NEXT token from the last position
    next_logits = logits[0, -1, :]       # (V,)
    probs       = F.softmax(next_logits, dim=-1)

    # Sample one token
    next_token  = torch.multinomial(probs, num_samples=1).item()

    # Log-prob of the token we actually sampled
    log_prob    = F.log_softmax(next_logits, dim=-1)[next_token]

    generated_tokens.append(next_token)
    log_probs_sampled.append(log_prob)
    all_logits_list.append(next_logits.detach())

    print(f"  t={step}  context='{tokens_to_str(current_ids[0].tolist())}'")
    print(f"       probs = { {VOCAB[i]: f'{probs[i].item():.3f}' for i in range(V)} }")
    print(f"       sampled → token {next_token} ('{VOCAB[next_token]}')   "
          f"log_prob = {log_prob.item():.4f}")
    print()

    # Extend context
    next_tensor = torch.tensor([[next_token]])
    current_ids = torch.cat([current_ids, next_tensor], dim=1)

    if next_token == EOS_ID:
        print("  [EOS reached, stopping]")
        break

response_str = tokens_to_str(generated_tokens)
print(f"  Generated response : {generated_tokens}  →  '{response_str}'")
print()


  t=0  context='a b'
       probs = {'PAD': '0.183', 'a': '0.102', 'b': '0.088', 'c': '0.057', 'd': '0.094', 'e': '0.095', 'f': '0.247', 'EOS': '0.133'}
       sampled → token 6 ('f')   log_prob = -1.3979

  t=1  context='a b f'
       probs = {'PAD': '0.201', 'a': '0.125', 'b': '0.210', 'c': '0.090', 'd': '0.086', 'e': '0.058', 'f': '0.098', 'EOS': '0.132'}
       sampled → token 7 ('EOS')   log_prob = -2.0263

  [EOS reached, stopping]
  Generated response : [6, 7]  →  'f EOS'



In [9]:
# ─────────────────────────────────────────────
# 4. Reward  (rule-based, fully transparent)
# ─────────────────────────────────────────────
print(SEP)
print("STEP 2 — REWARD")
print(SEP)

TARGET_TOKEN = 3   # "c"
R = sum(1.0 for t in generated_tokens if t == TARGET_TOKEN)

print(f"  Rule  : +1 for each '{VOCAB[TARGET_TOKEN]}' in the response")
print(f"  Count of '{VOCAB[TARGET_TOKEN]}' in {generated_tokens} : {int(R)}")
print(f"  R(τ)  = {R}")
print()

────────────────────────────────────────────────────────────
STEP 2 — REWARD
────────────────────────────────────────────────────────────
  Rule  : +1 for each 'c' in the response
  Count of 'c' in [6, 7] : 0
  R(τ)  = 0



In [10]:
# ─────────────────────────────────────────────
# 5. REINFORCE gradient  (no baseline first)
# ─────────────────────────────────────────────
print(SEP)
print("STEP 3 — REINFORCE GRADIENT (no baseline)")
print(SEP)
print("  ∇J(θ) ≈ R(τ) · Σ_t ∇ log π_θ(a_t | s_t)")
print()
print("  The scalar R(τ) multiplies the gradient of EVERY log-prob.")
print()

# Stack log-probs into a tensor so we can call .backward()
log_probs_tensor = torch.stack(log_probs_sampled)   # (T_gen,)
print(f"  log π_θ(a_t|s_t) per step : {[f'{lp.item():.4f}' for lp in log_probs_tensor]}")
print(f"  R(τ) = {R}")
print()

# The REINFORCE loss (we minimise -J)
loss_no_baseline = -R * log_probs_tensor.sum()
print(f"  loss = -R · Σ log_prob = -{R} × ({log_probs_tensor.sum().item():.4f})")
print(f"       = {loss_no_baseline.item():.4f}")
print()
print("  Problem: if R is always positive (as here), we ALWAYS push up")
print("  every action, even bad ones. High variance.")
print()


────────────────────────────────────────────────────────────
STEP 3 — REINFORCE GRADIENT (no baseline)
────────────────────────────────────────────────────────────
  ∇J(θ) ≈ R(τ) · Σ_t ∇ log π_θ(a_t | s_t)

  The scalar R(τ) multiplies the gradient of EVERY log-prob.

  log π_θ(a_t|s_t) per step : ['-1.3979', '-2.0263']
  R(τ) = 0

  loss = -R · Σ log_prob = -0 × (-3.4243)
       = -0.0000

  Problem: if R is always positive (as here), we ALWAYS push up
  every action, even bad ones. High variance.



In [11]:
# ─────────────────────────────────────────────
# 6. Baseline  b = mean reward over a mini-batch
# ─────────────────────────────────────────────
print(SEP)
print("STEP 4 — BASELINE SUBTRACTION")
print(SEP)

# Simulate a tiny mini-batch of rewards from other rollouts of the same prompt.
# In practice you'd generate multiple sequences; here we just illustrate
# with plausible values.
simulated_batch_rewards = torch.tensor([0.0, 1.0, 0.0, 2.0, 1.0, 0.0])
b = simulated_batch_rewards.mean().item()

print(f"  Simulated batch rewards : {simulated_batch_rewards.tolist()}")
print(f"  Baseline  b = mean(batch) = {b:.4f}")
print()

advantage = R - b
print(f"  Advantage  A = R(τ) - b = {R} - {b:.4f} = {advantage:.4f}")
print()

if advantage > 0:
    print("  A > 0 → this rollout was BETTER than average → push actions UP")
elif advantage < 0:
    print("  A < 0 → this rollout was WORSE  than average → push actions DOWN")
else:
    print("  A = 0 → exactly average → no gradient signal")
print()

print("  Why does this reduce variance?")
print("  Without baseline: gradient scale ~ R  (always positive, noisy)")
print("  With    baseline: gradient scale ~ A  (centered at 0, much smaller swings)")
print()

────────────────────────────────────────────────────────────
STEP 4 — BASELINE SUBTRACTION
────────────────────────────────────────────────────────────
  Simulated batch rewards : [0.0, 1.0, 0.0, 2.0, 1.0, 0.0]
  Baseline  b = mean(batch) = 0.6667

  Advantage  A = R(τ) - b = 0 - 0.6667 = -0.6667

  A < 0 → this rollout was WORSE  than average → push actions DOWN

  Why does this reduce variance?
  Without baseline: gradient scale ~ R  (always positive, noisy)
  With    baseline: gradient scale ~ A  (centered at 0, much smaller swings)



In [12]:
# ─────────────────────────────────────────────
# 7. One gradient step
# ─────────────────────────────────────────────
print(SEP)
print("STEP 5 — ONE GRADIENT UPDATE")
print(SEP)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# We need to recompute log-probs with grad enabled
# (the ones above were computed during model.eval() for inspection;
#  now we do a clean forward with gradients)

model.train()
optimizer.zero_grad()

current_ids_grad = torch.tensor([prompt])
log_probs_grad   = []

for t, action in enumerate(generated_tokens):
    logits      = model(current_ids_grad)
    next_logits = logits[0, -1, :]
    log_prob    = F.log_softmax(next_logits, dim=-1)[action]
    log_probs_grad.append(log_prob)
    next_tensor    = torch.tensor([[action]])
    current_ids_grad = torch.cat([current_ids_grad, next_tensor], dim=1)

log_probs_grad_tensor = torch.stack(log_probs_grad)

# REINFORCE with baseline loss
loss = -(advantage) * log_probs_grad_tensor.sum()

print(f"  loss = -A · Σ log_prob")
print(f"       = -({advantage:.4f}) × ({log_probs_grad_tensor.sum().item():.4f})")
print(f"       = {loss.item():.4f}")
print()

# Gradient of loss w.r.t. parameters
loss.backward()

# Show gradient norms per layer
print("  Gradient norms per parameter group:")
for name, param in model.named_parameters():
    if param.grad is not None:
        print(f"    {name:35s}  grad_norm = {param.grad.norm().item():.6f}")
print()

optimizer.step()
print("  optimizer.step() done — θ updated.")
print()

────────────────────────────────────────────────────────────
STEP 5 — ONE GRADIENT UPDATE
────────────────────────────────────────────────────────────
  loss = -A · Σ log_prob
       = -(-0.6667) × (-3.4243)
       = -2.2828

  Gradient norms per parameter group:
    embed.weight                         grad_norm = 0.276800
    pos_enc.weight                       grad_norm = 0.276800
    tf.layers.0.self_attn.in_proj_weight  grad_norm = 1.034832
    tf.layers.0.self_attn.in_proj_bias   grad_norm = 0.144155
    tf.layers.0.self_attn.out_proj.weight  grad_norm = 1.343607
    tf.layers.0.self_attn.out_proj.bias  grad_norm = 0.285300
    tf.layers.0.linear1.weight           grad_norm = 0.806422
    tf.layers.0.linear1.bias             grad_norm = 0.211261
    tf.layers.0.linear2.weight           grad_norm = 1.009109
    tf.layers.0.linear2.bias             grad_norm = 0.498070
    tf.layers.0.norm1.weight             grad_norm = 0.469454
    tf.layers.0.norm1.bias               grad_norm 

In [13]:
# ─────────────────────────────────────────────
# 8. Verify: did the update move in the right direction?
# ─────────────────────────────────────────────
print(SEP)
print("STEP 6 — VERIFICATION")
print(SEP)
print("  Recompute log-probs of the same response AFTER the update.")
print("  If A > 0, they should INCREASE. If A < 0, they should DECREASE.")
print()

model.eval()
current_ids_verify = torch.tensor([prompt])
log_probs_after    = []

with torch.no_grad():
    for action in generated_tokens:
        logits      = model(current_ids_verify)
        next_logits = logits[0, -1, :]
        log_prob    = F.log_softmax(next_logits, dim=-1)[action].item()
        log_probs_after.append(log_prob)
        current_ids_verify = torch.cat(
            [current_ids_verify, torch.tensor([[action]])], dim=1)

print(f"  {'token':>6}  {'before':>10}  {'after':>10}  {'Δ':>10}")
print(f"  {'─'*6}  {'─'*10}  {'─'*10}  {'─'*10}")
for t, (action, lp_before, lp_after) in enumerate(
        zip(generated_tokens,
            [lp.item() for lp in log_probs_grad],
            log_probs_after)):
    delta = lp_after - lp_before
    direction = "↑" if delta > 0 else "↓"
    print(f"  t={t}  '{VOCAB[action]}' ({action})  "
          f"{lp_before:10.4f}  {lp_after:10.4f}  {delta:+10.4f} {direction}")

print()
expected = "INCREASE" if advantage > 0 else ("DECREASE" if advantage < 0 else "UNCHANGED")
print(f"  Advantage was {advantage:.4f} → expected {expected}")
print()

────────────────────────────────────────────────────────────
STEP 6 — VERIFICATION
────────────────────────────────────────────────────────────
  Recompute log-probs of the same response AFTER the update.
  If A > 0, they should INCREASE. If A < 0, they should DECREASE.

   token      before       after           Δ
  ──────  ──────────  ──────────  ──────────
  t=0  'f' (6)     -1.3979     -1.4507     -0.0528 ↓
  t=1  'EOS' (7)     -2.0263     -2.1111     -0.0847 ↓

  Advantage was -0.6667 → expected DECREASE



In [14]:
# ─────────────────────────────────────────────
# 9. Conceptual summary
# ─────────────────────────────────────────────
print(SEP)
print("CONCEPTUAL SUMMARY")
print(SEP)
print("""
  1. SAMPLE     τ = (a_1, ..., a_T) autoregressively from π_θ
                Each a_t is one token. One "episode" = one response.

  2. REWARD     R(τ) is a scalar for the WHOLE sequence.
                (sparse: 0 at every step, only non-zero at the end)

  3. LOG-PROBS  log π_θ(a_t | s_t)  for each token sampled.
                Gradient flows through these.

  4. REINFORCE  loss = -R(τ) · Σ_t log π_θ(a_t | s_t)
                All tokens get the same reward signal.
                High variance because R is always ≥ 0.

  5. BASELINE   loss = -A · Σ_t log π_θ(a_t | s_t)
                A = R - b,  b = mean reward of the batch.
                Centered around 0 → same gradient direction in expectation,
                much lower variance in practice.

  6. PPO adds   - Ratio clipping: prevents too-large steps
                - A critic V(s_t): estimates b per-token (GAE)
                - KL penalty: don't stray from reference model

  What's the "action" here?
  → ONE TOKEN per step.  But the reward only arrives at the END.
  → Credit assignment problem: which token caused the high reward?
  → REINFORCE: all tokens equally. GAE: use V(s_t) to distribute credit.
""")

────────────────────────────────────────────────────────────
CONCEPTUAL SUMMARY
────────────────────────────────────────────────────────────

  1. SAMPLE     τ = (a_1, ..., a_T) autoregressively from π_θ
                Each a_t is one token. One "episode" = one response.

  2. REWARD     R(τ) is a scalar for the WHOLE sequence.
                (sparse: 0 at every step, only non-zero at the end)

  3. LOG-PROBS  log π_θ(a_t | s_t)  for each token sampled.
                Gradient flows through these.

  4. REINFORCE  loss = -R(τ) · Σ_t log π_θ(a_t | s_t)
                All tokens get the same reward signal.
                High variance because R is always ≥ 0.

  5. BASELINE   loss = -A · Σ_t log π_θ(a_t | s_t)
                A = R - b,  b = mean reward of the batch.
                Centered around 0 → same gradient direction in expectation,
                much lower variance in practice.

  6. PPO adds   - Ratio clipping: prevents too-large steps
                - A critic V(s_t):